# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [2]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


In [3]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [4]:
# Use esta célula para criar sua análise exploratória.

colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()

,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


In [5]:
# Preencha com uma variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

variavel_x = "taxa_abandono_carrinho_pct"

if variavel_x not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x,
    y="taxa_conversao_pct",
    title="Relação com a taxa de conversão",
    trendline="ols",
    labels={
        variavel_x: "Taxa de Abandono do Carrinho (%)",
        "taxa_conversao_pct": "Taxa de Conversão (%)",
    },
)
fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Escolhi `taxa_abandono_carrinho_pct` e `profundidade_scroll_pct`.

O abandono de carrinho foi a escolha mais óbvia, já que a correlação de -0.643 com a conversão é bem mais forte que as outras. Faz sentido também pelo contexto: quem abandona o carrinho não converte.

O scroll entrou no lugar do tempo de clique porque o r de +0.485 é consideravelmente maior que o -0.229 do tempo. Descartei `tempo_primeiro_clique_s` não só pelo número menor, mas porque olhando o scatter a relação parecia bem mais espalhada, menos clara visualmente.

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Scatter plots das 3 features vs taxa de conversão ---
nomes_variaveis = {
    'taxa_abandono_carrinho_pct': 'Taxa de Abandono do Carrinho (%)',
    'profundidade_scroll_pct': 'Profundidade de Scroll (%)',
    'tempo_primeiro_clique_s': 'Tempo até 1º Clique (s)',
}

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[nomes_variaveis[f] for f in features],
    shared_yaxes=True,
)

cores = ['#EF553B', '#00CC96', '#636EFA']
for i, feature in enumerate(features):
    fig.add_trace(
        go.Scatter(
            x=df[feature],
            y=df[target],
            mode='markers',
            marker=dict(color=cores[i], opacity=0.6, size=6),
            name=nomes_variaveis[feature],
            showlegend=False,
        ),
        row=1, col=i + 1,
    )
    # linha de tendência linear
    x_vals = df[feature].values
    y_vals = df[target].values
    m, b = np.polyfit(x_vals, y_vals, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    fig.add_trace(
        go.Scatter(
            x=x_line, y=m * x_line + b,
            mode='lines',
            line=dict(color=cores[i], width=2, dash='dash'),
            showlegend=False,
        ),
        row=1, col=i + 1,
    )

fig.update_yaxes(title_text='Taxa de Conversão (%)', row=1, col=1)
fig.update_layout(
    title_text='Relação de cada variável com a Taxa de Conversão',
    height=400,
)
fig.show()

# --- Heatmap de correlação ---
corr_matrix = df[colunas_numericas].corr()
fig_corr = px.imshow(
    corr_matrix,
    text_auto='.3f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Matriz de Correlação',
    aspect='auto',
)
fig_corr.show()

# --- Resumo das correlações com a taxa de conversão ---
corr_target = (
    df[colunas_numericas].corr()[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
)
print('Correlações com taxa_conversao_pct (ordem decrescente de força):')
print(corr_target.to_string())

Correlações com taxa_conversao_pct (ordem decrescente de força):
taxa_abandono_carrinho_pct   -0.643
profundidade_scroll_pct       0.485
tempo_primeiro_clique_s      -0.229


## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [7]:
X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

,métrica,valor
0,MAE,0.276
1,RMSE,0.344


### Análise Complementar: Coeficientes Padronizados (Beta Normalizados)

Para comparar o impacto relativo de variáveis em escalas diferentes, calculamos os **coeficientes padronizados** (beta), que medem a variação na saída em desvios-padrão para cada desvio-padrão de mudança na entrada. Isso elimina o efeito das unidades de medida.

In [8]:
# Coeficientes padronizados (beta normalizados)
# Fórmula: beta_i = coef_i * (std_Xi / std_y)

# Coeficientes do modelo (índices 1, 2, 3 = features; 0 = intercepto)
print('Coeficientes brutos do modelo:')
for fname, coef in zip(features, coeficientes[1:]):
    print(f'  {fname}: {coef:.6f}')

std_X = df[features].std()
std_y = df[target].std()

betas = coeficientes[1:] * std_X.values / std_y

df_betas = pd.DataFrame({
    'variável': features,
    'coeficiente_bruto': coeficientes[1:],
    'std_variável': std_X.values,
    'beta_padronizado': betas,
    'impacto_abs': abs(betas),
}).sort_values('impacto_abs', ascending=False)

print('\nCoeficientes padronizados (beta):')
print(df_betas[['variável', 'coeficiente_bruto', 'beta_padronizado']].to_string(index=False))

# Gráfico
fig_betas = px.bar(
    df_betas,
    x='beta_padronizado',
    y='variável',
    orientation='h',
    color='beta_padronizado',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    text='beta_padronizado',
    title='Coeficientes Padronizados (β) — Importância relativa das variáveis',
    labels={'beta_padronizado': 'β (desvios-padrão)', 'variável': ''},
)
fig_betas.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_betas.update_layout(coloraxis_showscale=False, height=300)
fig_betas.show()

Coeficientes brutos do modelo:
  taxa_abandono_carrinho_pct: -0.060372
  profundidade_scroll_pct: 0.024226
  tempo_primeiro_clique_s: -0.094151

Coeficientes padronizados (beta):
                  variável  coeficiente_bruto  beta_padronizado
taxa_abandono_carrinho_pct             -0.060            -0.650
   profundidade_scroll_pct              0.024             0.457
   tempo_primeiro_clique_s             -0.094            -0.328


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

**Resposta:**

O MAE ficou em 0.276 e o RMSE em 0.344, os dois em pontos percentuais. A conversão no dataset varia de 4.35% a 7.75%, uma amplitude de uns 3.4 p.p., então errar em média 0.276 p.p. parece razoável, cerca de 8% da faixa total.

O RMSE um pouco maior que o MAE sugere que tem alguns dias com erro acima da média, mas nada extremo. Para o que a atividade propõe, que é comparar sensibilidades e não fazer previsão de alta precisão, esse ajuste é suficiente.

## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [9]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [10]:
# Preencha com duas variáveis escolhidas na Parte 1.
# Use exatamente os nomes que aparecem em features.

variaveis_escolhidas = ["taxa_abandono_carrinho_pct", "profundidade_scroll_pct"]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

In [11]:
# Visualização dos índices de sensibilidade
fig_sens = px.bar(
    tabela_sensibilidade,
    x='variável',
    y='índice_sensibilidade',
    color='índice_sensibilidade',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    text='índice_sensibilidade',
    title='Índice de Sensibilidade por Variável (variação de +10% na entrada)',
    labels={
        'variável': 'Variável de Entrada',
        'índice_sensibilidade': 'Índice de Sensibilidade',
    },
)
fig_sens.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig_sens.update_layout(coloraxis_showscale=False, yaxis_range=[-0.7, 0.5])
fig_sens.show()

print(tabela_sensibilidade[['variável', 'índice_sensibilidade']].to_string(index=False))

                  variável  índice_sensibilidade
taxa_abandono_carrinho_pct                -0.489
   profundidade_scroll_pct                 0.258


**Resposta:**

Olhando a tabela, o abandono de carrinho saiu bem na frente: índice de -0.489 contra +0.258 do scroll. Em termos práticos, aumentar o abandono em 10% derruba a conversão quase 4.9%, enquanto o mesmo aumento percentual no scroll levanta só 2.6%.

A diferença em módulo é considerável, quase o dobro. Então se a equipe tiver que escolher uma variável para trabalhar, o abandono de carrinho tem mais retorno potencial por esforço. O sinal negativo já era esperado: mais abandono, menos conversão. O scroll positivo também faz sentido, usuário que scrolla mais está explorando o catálogo.

### Análise Complementar: Tornado Chart (todas as 3 variáveis)

O **Tornado Chart** é a visualização padrão em análise de sensibilidade. Ele mostra o intervalo de variação da saída quando cada variável oscila ±10% a partir da linha de base, com as barras ordenadas da maior à menor sensibilidade.

In [12]:
# Tornado Chart — sensibilidade de todas as 3 variáveis (±10%)

variacao = 0.10
resultados_tornado = []

for variavel in features:
    valor_base = linha_base[variavel]

    # Cenário otimista: variação que MELHORA a conversão
    linha_mais = linha_base.copy()
    linha_menos = linha_base.copy()
    linha_mais[variavel] = valor_base * (1 + variacao)
    linha_menos[variavel] = valor_base * (1 - variacao)

    saida_mais = prever_linha(linha_mais)
    saida_menos = prever_linha(linha_menos)

    # Índice de sensibilidade (usando +10%)
    is_val = ((saida_mais - saida_base) / saida_base) / variacao

    resultados_tornado.append({
        'variável': variavel,
        'saída_−10%': min(saida_mais, saida_menos),
        'saída_+10%': max(saida_mais, saida_menos),
        'amplitude': abs(saida_mais - saida_menos),
        'índice_sensibilidade': is_val,
    })

df_tornado = pd.DataFrame(resultados_tornado).sort_values('amplitude', ascending=True)

nomes_curtos = {
    'taxa_abandono_carrinho_pct': 'Abandono de Carrinho',
    'profundidade_scroll_pct': 'Profundidade de Scroll',
    'tempo_primeiro_clique_s': 'Tempo até 1º Clique',
}

import plotly.graph_objects as go

fig_tornado = go.Figure()

for _, row in df_tornado.iterrows():
    nome = nomes_curtos.get(row['variável'], row['variável'])
    # Barra da faixa de variação
    fig_tornado.add_trace(go.Bar(
        y=[nome],
        x=[row['saída_+10%'] - row['saída_−10%']],
        base=[row['saída_−10%']],
        orientation='h',
        marker_color='#3B82F6' if row['índice_sensibilidade'] < 0 else '#10B981',
        showlegend=False,
        text=f"±{row['amplitude']:.3f} p.p.",
        textposition='outside',
    ))

fig_tornado.add_vline(
    x=saida_base,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'Base: {saida_base:.3f}%',
    annotation_position='top',
)

fig_tornado.update_layout(
    title='Tornado Chart — Impacto de ±10% em cada variável sobre a taxa de conversão',
    xaxis_title='Taxa de Conversão Prevista (%)',
    yaxis_title='',
    height=350,
    bargap=0.4,
)
fig_tornado.show()

print('\nAmplitude de variação por variável (±10%):')
print(df_tornado[['variável', 'saída_−10%', 'saída_+10%', 'amplitude', 'índice_sensibilidade']].to_string(index=False))


Amplitude de variação por variável (±10%):
                  variável  saída_−10%  saída_+10%  amplitude  índice_sensibilidade
   tempo_primeiro_clique_s       5.803       5.934      0.131                -0.112
   profundidade_scroll_pct       5.717       6.020      0.303                 0.258
taxa_abandono_carrinho_pct       5.582       6.156      0.574                -0.489


## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

A recomendação é focar em reduzir o abandono de carrinho no próximo ciclo. O índice de -0.489 mostrou que essa variável move mais a conversão que o scroll, e a relação faz sentido operacionalmente porque carrinho abandonado é receita que saiu da mesa.

Uma redução de 10% no abandono, saindo de 47.5% para perto de 42.8%, levaria a conversão de 5.87% para algo em torno de 6.16% pela estimativa do modelo. Não é uma mudança enorme em absoluto, mas dentro da faixa histórica dos dados é relevante.

Em termos de ação concreta, simplificar o checkout e salvar o carrinho entre sessões parecem os caminhos mais diretos. O scroll também pode ser trabalhado depois, mas o impacto esperado é menor.

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

A principal limitação que vejo é a hipótese de linearidade. O modelo assume que cada variável contribui de forma proporcional e independente para a conversão, mas no mundo real provavelmente tem interações. Por exemplo, talvez o scroll só ajude quando o abandono já está controlado, ou o efeito diminua acima de um certo threshold.

Além disso, os dados são simulados com seed fixa, o que significa que as correlações já estavam "plantadas" no processo gerador. Em dados reais haveria mais ruído e possivelmente relações diferentes dependendo do período ou do perfil de usuário.

In [13]:
# Visualização auxiliar: impacto simulado de reduzir o abandono de carrinho
reducoes = np.arange(0, 0.31, 0.05)  # de 0% a 30% de redução
conversoes_simuladas = []
for reducao in reducoes:
    linha_sim = linha_base.copy()
    linha_sim['taxa_abandono_carrinho_pct'] *= (1 - reducao)
    conversoes_simuladas.append(prever_linha(linha_sim))

fig_decisao = px.line(
    x=reducoes * 100,
    y=conversoes_simuladas,
    markers=True,
    labels={
        'x': 'Redução no Abandono de Carrinho (%)',
        'y': 'Taxa de Conversão Prevista (%)',
    },
    title='Impacto simulado da redução do abandono de carrinho na taxa de conversão',
)
fig_decisao.add_hline(
    y=saida_base,
    line_dash='dash',
    line_color='gray',
    annotation_text=f'Base: {saida_base:.3f}%',
    annotation_position='right',
)
fig_decisao.show()

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [14]:
# Use esta célula para sua simulação.

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

,0
count,1000.000
mean,5.869
std,0.393
min,4.654
10%,5.369
25%,5.617
50%,5.854
75%,6.131
90%,6.379
max,7.193


In [15]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

### Análise Complementar: Quantificação de Risco e Sensibilidade Global (Monte Carlo)

Além da distribuição geral, podemos usar os dados simulados para responder perguntas de negócio específicas e identificar qual variável contribui mais para a **incerteza total** da previsão.

In [16]:
# Análise de risco quantitativa a partir da simulação de Monte Carlo

limiar_risco = 5.5    # % — abaixo disso, conversão preocupante
limiar_oport = 6.3    # % — acima disso, conversão excelente

prob_risco = (previsoes < limiar_risco).mean() * 100
prob_oport = (previsoes > limiar_oport).mean() * 100

print('=== Análise de Risco (Monte Carlo, n=1000) ===')
print(f'P(conversão < {limiar_risco}%) = {prob_risco:.1f}%  ← risco de desempenho ruim')
print(f'P(conversão > {limiar_oport}%) = {prob_oport:.1f}%  ← probabilidade de desempenho excelente')
print(f'Intervalo de confiança 80%: [{pd.Series(previsoes).quantile(0.10):.3f}%, {pd.Series(previsoes).quantile(0.90):.3f}%]')

# Histograma enriquecido com zonas de risco
import plotly.graph_objects as go

fig_mc = go.Figure()
fig_mc.add_trace(go.Histogram(
    x=previsoes,
    nbinsx=40,
    name='Distribuição simulada',
    marker_color='#6366F1',
    opacity=0.8,
))

# Zona de risco
fig_mc.add_vrect(x0=previsoes.min()-0.1, x1=limiar_risco,
    fillcolor='red', opacity=0.1, line_width=0,
    annotation_text='Zona de risco', annotation_position='top left')

# Zona de oportunidade
fig_mc.add_vrect(x0=limiar_oport, x1=previsoes.max()+0.1,
    fillcolor='green', opacity=0.1, line_width=0,
    annotation_text='Zona ótima', annotation_position='top right')

# Linha da base
fig_mc.add_vline(x=saida_base, line_dash='dash', line_color='orange',
    annotation_text=f'Base: {saida_base:.3f}%')

fig_mc.update_layout(
    title='Distribuição Monte Carlo — Zonas de Risco e Oportunidade',
    xaxis_title='Taxa de Conversão Prevista (%)',
    yaxis_title='Frequência',
    height=400,
)
fig_mc.show()

# Análise de variância explicada por variável (sensibilidade global)
print('\n=== Contribuição de cada variável para a variância total da previsão ===')
variancia_total = pd.Series(previsoes).var()

for feat in features:
    # Simula fixando cada variável na média, varia apenas as outras
    amostras_fixo = amostras.copy()
    amostras_fixo[feat] = linha_base[feat]  # fixar no valor base
    design_fixo = np.column_stack([np.ones(len(amostras_fixo)), amostras_fixo[features].to_numpy()])
    prev_fixo = design_fixo @ coeficientes
    var_sem = pd.Series(prev_fixo).var()
    reducao_var = (variancia_total - var_sem) / variancia_total * 100
    print(f'  {feat}: contribui com ~{reducao_var:.1f}% da variância total')

=== Análise de Risco (Monte Carlo, n=1000) ===
P(conversão < 5.5%) = 17.5%  ← risco de desempenho ruim
P(conversão > 6.3%) = 13.8%  ← probabilidade de desempenho excelente
Intervalo de confiança 80%: [5.369%, 6.379%]



=== Contribuição de cada variável para a variância total da previsão ===
  taxa_abandono_carrinho_pct: contribui com ~64.2% da variância total
  profundidade_scroll_pct: contribui com ~26.9% da variância total
  tempo_primeiro_clique_s: contribui com ~10.0% da variância total


Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

**Resposta:**

A distribuição saiu bastante concentrada: média de 5.87%, desvio padrão de 0.393 p.p. O intervalo P10 a P90 vai de 5.37% a 6.38%, o que significa que em 80% dos cenários simulados a conversão fica nessa faixa de pouco mais de 1 ponto percentual.

O que isso diz sobre o risco da recomendação é que, mesmo com variabilidade natural nas entradas, a conversão raramente cai muito longe da base. O pior cenário simulado ficou em 4.65%, e o melhor em 7.19%. Isso dá uma segurança razoável para apostar na melhoria do abandono de carrinho, já que o downside simulado é limitado.

## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.